# AnonyMED-BR — LLM-to-encoder knowledge distillation

## Clinical de-identification in Brazilian Portuguese: full experimental pipeline



### Hardware

Teacher generation was run on an A100 40GB; the student trains on any modern
GPU; latency is measured on CPU and the CPU model is recorded in the manifest.

---
## 1. Environment and configuration

In [ ]:
# !pip install -q transformers accelerate datasets seqeval spacy scipy pandas matplotlib
# !python -m spacy download pt_core_news_sm
import os, sys, json, time, math, random, hashlib, platform, subprocess, warnings
from dataclasses import dataclass, field, asdict
from pathlib import Path
from typing import Dict, List, Sequence, Tuple, Optional

import gc
import numpy as np, pandas as pd, torch
import torch.nn as nn, torch.nn.functional as F

warnings.filterwarnings("ignore", category=UserWarning)
try:
    from transformers.utils import logging as hf_logging
    hf_logging.set_verbosity_error()      # silences the per-run LOAD REPORT
except Exception:
    pass

def env_manifest() -> dict:
    m = {
        "python": sys.version.split()[0],
        "platform": platform.platform(),
        "cpu_model": platform.processor() or "unknown",
        "cpu_count_logical": os.cpu_count(),
        "torch": torch.__version__,
        "cuda_available": torch.cuda.is_available(),
    }
    try:
        import transformers; m["transformers"] = transformers.__version__
    except ImportError: m["transformers"] = None
    try:
        import seqeval; m["seqeval"] = getattr(seqeval, "__version__", "installed")
    except ImportError: m["seqeval"] = None
    if torch.cuda.is_available():
        m["gpu"] = torch.cuda.get_device_name(0)
        m["gpu_mem_gb"] = round(torch.cuda.get_device_properties(0).total_memory / 1e9, 1)
    # CPU model name is reported alongside the latency measurements
    try:
        out = subprocess.run(["lscpu"], capture_output=True, text=True, timeout=5).stdout
        for line in out.splitlines():
            if "Model name" in line:
                m["cpu_model"] = line.split(":", 1)[1].strip(); break
    except Exception: pass
    return m

ENV = env_manifest()
for k, v in ENV.items(): print(f"{k:22s} {v}")

In [ ]:
@dataclass
class Config:
    # --- data -------------------------------------------------------------
    data_dir: str = "."
    from_hub: bool = False              # True -> load_dataset("Venturus/AnonyMED-BR")
    granularity: str = "sub"            # "sub" (19 subcats, 39 labels) | "coarse" (8, 17)
    window_size: int = 200              # official train window (spaCy tokens)
    overlap: int = 50                   # official train overlap
    eval_window: int = 150              # official test window
    eval_overlap: int = 0
    max_length: int = 512               # WordPiece length for the student

    # --- teacher ----------------------------------------------------------
    teacher: str = "Qwen/Qwen2.5-7B-Instruct"
    teacher_dtype: str = "bfloat16"     # A100 40GB fits 7B in bf16; no NF4 needed
    teacher_batch: int = 8
    teacher_max_words: int = 200        # must be >= window_size or fallback explodes

    # --- student ----------------------------------------------------------
    student: str = "neuralmind/bert-base-portuguese-cased"
    epochs: int = 5
    batch_size: int = 16
    warmup_ratio: float = 0.1
    weight_decay: float = 0.01

    # --- KD ---------------------------------------------------------------
    kd_T: float = 4.0
    kd_alpha: float = 0.5

    # --- protocol ---------------------------------------------------------
    seeds: Tuple[int, ...] = (13, 21, 42, 87, 101)
    bootstrap_n: int = 10_000
    permutation_n: int = 10_000
    out_dir: str = "./outputs"

    # --- smoke test -------------------------------------------------------
    SMOKE: bool = False                 # True -> tiny subset, full code path, ~10 min
    smoke_docs: int = 24
    allow_weak_teacher: bool = False    # True -> proceed and report weak distillation

    def __post_init__(self):
        Path(self.out_dir).mkdir(parents=True, exist_ok=True)
        if self.SMOKE:
            self.epochs, self.seeds = 1, (42,)
            self.bootstrap_n = self.permutation_n = 200

CFG = Config()
print(f"SMOKE={CFG.SMOKE}  granularity={CFG.granularity}  seeds={CFG.seeds}")
print("Set SMOKE=True for a fast end-to-end validation run on a small subset;\n"
      "SMOKE=False is the configuration that produces the reported numbers.")

def set_seed(s: int):
    random.seed(s); np.random.seed(s); torch.manual_seed(s)
    torch.cuda.manual_seed_all(s)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False

---
## 2. Data — preprocessing with strict integrity checks


In [ ]:

import re
from collections import Counter

SUBCATEGORIES = ["AGE","PHONE","EMAIL","DATE","IDNUM","MEDICAL_RECORD","HEALTH_PLAN",
                 "STREET","CITY","ZIP","STATE","COUNTRY","LOCATION_OTHER",
                 "ORGANIZATION","HOSPITAL","PATIENT","DOCTOR","PROFESSION","OTHER"]
COARSE = ["NAME","LOCATION","CONTACT","DATE","ID","AGE","PROFESSION","OTHER"]
SUB2COARSE = {"PATIENT":"NAME","DOCTOR":"NAME",
              "CITY":"LOCATION","STREET":"LOCATION","HOSPITAL":"LOCATION","STATE":"LOCATION",
              "COUNTRY":"LOCATION","ORGANIZATION":"LOCATION","LOCATION_OTHER":"LOCATION",
              "PHONE":"CONTACT","EMAIL":"CONTACT","DATE":"DATE",
              "IDNUM":"ID","MEDICAL_RECORD":"ID","ZIP":"ID","HEALTH_PLAN":"ID",
              "AGE":"AGE","PROFESSION":"PROFESSION","OTHER":"OTHER"}
TAG_RE = re.compile(r"</?([A-Z_]+)/?>")

def build_labels(gran: str) -> List[str]:
    types = SUBCATEGORIES if gran == "sub" else COARSE
    return ["O"] + [f"{p}-{t}" for t in types for p in ("B", "I")]

LABELS   = build_labels(CFG.granularity)
LABEL2ID = {l: i for i, l in enumerate(LABELS)}
ID2LABEL = {i: l for l, i in LABEL2ID.items()}
N_LABELS = len(LABELS)
print(f"{N_LABELS} BIO labels ({CFG.granularity})")

def split_tags(text: str) -> str:
    """Official convention: structured entities become one tag per token."""
    text = re.sub(r"<DATE>(\d{2})/(\d{2})/(\d{2,4})</DATE/>",
                  r"<DATE>\1</DATE/> <DATE>/</DATE/> <DATE>\2</DATE/> "
                  r"<DATE>/</DATE/> <DATE>\3</DATE/>", text)
    text = re.sub(r"<DATE>(\d{2})/(\d{2,4})</DATE/>",
                  r"<DATE>\1</DATE/> <DATE>/</DATE/> <DATE>\2</DATE/>", text)
    text = re.sub(r"<PHONE>\((\d{2})\)</PHONE/>", r"( <PHONE>\1</PHONE/> )", text)
    text = re.sub(r"<PHONE>(\d{4,5})-(\d{4})</PHONE/>",
                  r"<PHONE>\1</PHONE/> <PHONE>-</PHONE/> <PHONE>\2</PHONE/>", text)
    text = re.sub(r"<EMAIL>([\w\.-]+)@([\w\.-]+)</EMAIL/>",
                  r"<EMAIL>\1</EMAIL/> <EMAIL>@</EMAIL/> <EMAIL>\2</EMAIL/>", text)
    return text

_ALL_TAGS = SUBCATEGORIES + COARSE + ["LOCATION", "NAME", "CONTACT", "ID"]
def fix_tags(text: str) -> str:
    for t in _ALL_TAGS:
        for a, b in ((f"< {t} >", f"<{t}>"), (f"< {t}>", f"<{t}>"), (f"<{t} >", f"<{t}>"),
                     (f"</ {t} >", f"</{t}/>"), (f"</ {t}>", f"</{t}/>"), (f"</{t} >", f"</{t}/>"),
                     (f"<{t}/ >", f"</{t}/>"), (f"</{t}/ >", f"</{t}/>"),
                     (f"<{t}> ", f"<{t}>"), (f" </{t}>", f"</{t}/>"), (f"</{t}>", f"</{t}/>")):
            text = text.replace(a, b)
    return text

_NLP = None
def nlp():
    """spaCy tokenizer, loaded once.

    Prefers pt_core_news_sm; falls back to blank('pt'), which produces IDENTICAL
    tokenisation (the official pipeline only uses token.text) and needs no model
    download. Verified token-by-token on 40 documents.
    """
    global _NLP
    if _NLP is None:
        import spacy
        try:
            _NLP = spacy.load("pt_core_news_sm",
                              disable=["parser","ner","lemmatizer","tagger","attribute_ruler"])
            print("spaCy: pt_core_news_sm")
        except OSError:
            _NLP = spacy.blank("pt")
            print("spaCy: blank('pt') -- identical tokenisation, no download")
        _NLP.max_length = 2_000_000
    return _NLP



KNOWN_TYPES = set(SUBCATEGORIES) | set(COARSE)
ORPHAN_TAGS = Counter()     # tags present in the text but absent from `labels`

def detag(text: str) -> Tuple[str, List[Tuple[int, int, str]]]:
    parts, spans, out, open_type, open_start, i = [], [], 0, None, None, 0
    for m in TAG_RE.finditer(text):
        parts.append(text[i:m.start()]); out += m.start() - i; i = m.end()
        raw, typ = m.group(0), m.group(1)
        if raw.startswith("</"):
            if open_type == typ: spans.append((open_start, out, typ))
            open_type = None
        else:
            open_type, open_start = typ, out
    parts.append(text[i:])
    return "".join(parts), spans

def tokenize_with_types(text: str) -> Tuple[List[str], List[Optional[str]]]:
    clean, spans = detag(text)
    doc = nlp()(clean)
    toks = [t.text for t in doc]
    types: List[Optional[str]] = [None] * len(doc)
    for s, e, typ in spans:
        if typ not in KNOWN_TYPES:
            ORPHAN_TAGS[typ] += 1          # e.g. a stray <TIME> with no label entry
            continue
        for k, t in enumerate(doc):
            if t.idx < e and t.idx + len(t.text) > s:
                types[k] = typ
    return toks, types

def windows(toks, types, size, overlap):
    step = size - overlap
    assert step > 0, "overlap must be < window_size"
    out = []
    for i in range(0, len(toks), step):
        out.append((toks[i:i+size], types[i:i+size]))
        if i + size >= len(toks): break
    return out

def to_bio(types: Sequence[Optional[str]]) -> List[str]:
    tags, prev = [], None
    for ty in types:
        if ty is None:
            tags.append("O"); prev = None; continue
        t = SUB2COARSE.get(ty, "OTHER") if CFG.granularity == "coarse" else ty
        tags.append(f"{'I' if t == prev else 'B'}-{t}"); prev = t
    return tags

def assert_clean(toks, tags, where=""):
    if len(toks) != len(tags):
        raise AssertionError(f"{where}: {len(toks)} tokens vs {len(tags)} tags")
    leak = [t for t in toks if TAG_RE.search(t)]
    if leak:
        raise AssertionError(
            f"{where}: annotation markup leaked into model input: {leak[:5]}\\n"
            "Training on this inflates F1 -- the model can read the answer.")
    unknown = set(tags) - set(LABELS)
    if unknown: raise AssertionError(f"{where}: labels outside inventory: {sorted(unknown)}")
    for i, t in enumerate(tags):
        if t.startswith("I-") and (tags[i-1][2:] if i else "") != t[2:]:
            raise AssertionError(f"{where}: orphan {t} at {i}")
print("preprocessing defined")

In [ ]:
def load_raw() -> Dict[str, List[dict]]:
    if CFG.from_hub:
        from datasets import load_dataset
        ds = load_dataset("Venturus/AnonyMED-BR")
        raw = {k: list(ds[k]) for k in ("train", "validation", "test")}
    else:
        d = Path(CFG.data_dir)
        raw = {"train":      json.loads((d/"clinical_deid_training_set.json").read_text("utf-8")),
               "validation": json.loads((d/"clinical_deid_validation_set.json").read_text("utf-8")),
               "test":       json.loads((d/"clinical_deid_test_set.json").read_text("utf-8"))}
    # Integrity: every annotation offset must match the raw text exactly.
    bad = sum(1 for s in raw for r in raw[s] for l in r["labels"]
              if r["text"][l["first_position"]:l["last_position"]] != l["word"])
    if bad: raise AssertionError(f"{bad} offset mismatches -- wrong dataset version")
    # Provenance: freeze what we actually read.
    for s, recs in raw.items():
        h = hashlib.sha256(json.dumps(recs, sort_keys=True).encode()).hexdigest()[:16]
        print(f"{s:11s} {len(recs):5d} docs  sha256:{h}  synthetic={set(r.get('synthetic') for r in recs)}")
    return raw

def build_corpus(raw, size=None, overlap=None) -> Dict[str, List[dict]]:
    size    = CFG.window_size if size    is None else size
    overlap = CFG.overlap     if overlap is None else overlap
    corpus = {}
    for split, recs in raw.items():
        if CFG.SMOKE: recs = recs[:CFG.smoke_docs]
        chunks = []
        for r in recs:
            text = " ".join(split_tags(r["text"]).replace("\n", " ").split()).strip()
            toks_all, types_all = tokenize_with_types(text)
            for w, (toks, types) in enumerate(windows(toks_all, types_all, size, overlap)):
                if not toks: continue
                tags = to_bio(types)
                assert_clean(toks, tags, f"{split}/doc{r['id']}/win{w}")
                chunks.append({"id": f"{r['id']}-{w}", "doc_id": r["id"],
                               "tokens": toks, "tags": tags,
                               "label_ids": [LABEL2ID[t] for t in tags]})
        corpus[split] = chunks
    ids = {s: {c["doc_id"] for c in v} for s, v in corpus.items()}
    for a, b in (("train","validation"), ("train","test"), ("validation","test")):
        if ids[a] & ids[b]: raise AssertionError(f"document leakage {a}/{b}")
    return corpus

def load_prebuilt(name: str) -> Optional[Dict[str, List[dict]]]:
    """Reuse the windows produced by anonymed_data.ipynb when available.

    Preferred path: the two notebooks then provably share one preprocessing.
    """
    f = Path(CFG.data_dir) / f"{name}_{CFG.granularity}.json"
    if not f.exists() or CFG.SMOKE: return None
    data = json.loads(f.read_text())
    for split, chunks in data.items():
        for c in chunks:
            c["tags"] = [ID2LABEL[i] for i in c["label_ids"]]
    print(f"loaded {f.name}: " + str({k: len(v) for k, v in data.items()}))
    return data

RAW = load_raw()
CORPUS      = load_prebuilt("train_windows") or build_corpus(RAW)
CORPUS_EVAL = load_prebuilt("eval_windows")  or build_corpus(RAW, CFG.eval_window, CFG.eval_overlap)
if ORPHAN_TAGS:
    print(f"[!] tags outside the official inventory, treated as O: {dict(ORPHAN_TAGS)}")
print("windows (train):", {k: len(v) for k, v in CORPUS.items()})
print("windows (eval) :", {k: len(v) for k, v in CORPUS_EVAL.items()})

---
## 3. Table 1 and entity distribution



In [ ]:
def spans_of(tags: Sequence[str]) -> List[Tuple[str, int]]:
    out = []
    for t in tags:
        if t.startswith("B-"): out.append([t[2:], 1])
        elif t.startswith("I-") and out: out[-1][1] += 1
    return [tuple(x) for x in out]

def table1(corpus) -> pd.DataFrame:
    rows = []
    for split, chunks in corpus.items():
        ntok = sum(len(c["tags"]) for c in chunks)
        sp   = [s for c in chunks for s in spans_of(c["tags"])]
        rows.append({"Split": split, "Documents": len({c["doc_id"] for c in chunks}),
                     "Windows": len(chunks), "Tokens": ntok, "Entities": len(sp),
                     "PHI density (%)": round(100*len(sp)/ntok, 2),
                     "Mean span (tok)": round(np.mean([l for _, l in sp]), 2)})
    df = pd.DataFrame(rows)
    df.loc[len(df)] = {"Split":"TOTAL","Documents":df.Documents.sum(),"Windows":df.Windows.sum(),
                       "Tokens":df.Tokens.sum(),"Entities":df.Entities.sum(),
                       "PHI density (%)":round(100*df.Entities.sum()/df.Tokens.sum(),2),
                       "Mean span (tok)":np.nan}
    return df

def entity_table(corpus, split="train") -> pd.DataFrame:
    sp = [s for c in corpus[split] for s in spans_of(c["tags"])]
    per = {}
    for t, l in sp: per.setdefault(t, Counter())[min(l, 4)] += 1
    tot = len(sp)
    rows = [{"Entity": t, "Count": sum(d.values()), "Share (%)": round(100*sum(d.values())/tot, 2),
             "1 tok": d[1], "2 tok": d[2], "3 tok": d[3], "4+ tok": d[4]}
            for t, d in sorted(per.items(), key=lambda kv: -sum(kv[1].values()))]
    return pd.DataFrame(rows)

# Corpus statistics use NON-OVERLAPPING windows: the 50-token training overlap
# would double-count entities and tokens (inflation of 7-9%).
T1 = table1(CORPUS_EVAL)
ENT_TRAIN = entity_table(CORPUS_EVAL, "train")
ENT_TEST  = entity_table(CORPUS_EVAL, "test")
print("=== Table 1 ==="); print(T1.to_string(index=False))
print("\n=== Entities (test) ==="); print(ENT_TEST.to_string(index=False))

low = ENT_TEST[ENT_TEST.Count < 10]
if len(low):
    print("\n[!] Types with <10 test instances -- no per-type performance claim is defensible:")
    print("    " + ", ".join(f"{r.Entity}({r.Count})" for r in low.itertuples()))

for df, name in ((T1,"table1"), (ENT_TRAIN,"entities_train"), (ENT_TEST,"entities_test")):
    df.to_csv(f"{CFG.out_dir}/{name}.csv", index=False)

---
## 4. Rule-based (regex) baseline


In [ ]:
RX = {
    "DATE_D":  re.compile(r"^\d{1,2}$"),
    "DATE_Y":  re.compile(r"^(19|20)\d{2}$"),
    "SEP":     re.compile(r"^[/\-.]$"),
    "PHONE_P": re.compile(r"^\d{4,5}$"),
    "PHONE_S": re.compile(r"^\d{4}$"),
    "DDD":     re.compile(r"^\d{2}$"),
    "EMAIL_U": re.compile(r"^[\w.\-]+$"),
    "EMAIL_D": re.compile(r"^[\w\-]+\.[a-z]{2,}(\.[a-z]{2})?$"),
    "IDNUM":   re.compile(r"^\d{6,}$"),
    "CPF":     re.compile(r"^\d{3}\.\d{3}\.\d{3}-\d{2}$"),
    "ZIP":     re.compile(r"^\d{5}-?\d{3}$"),
}
AGE_CUE = {"anos", "ano", "meses", "idade"}
_MAP = (lambda t: SUB2COARSE.get(t, "OTHER")) if CFG.granularity == "coarse" else (lambda t: t)

def regex_tag(tokens: List[str]) -> List[str]:
    """Deterministic rule tagger. Conservative: emits O when unsure."""
    n = len(tokens); tags = ["O"] * n; i = 0
    def put(a, b, typ):
        t = _MAP(typ)
        if f"B-{t}" not in LABEL2ID: return
        tags[a] = f"B-{t}"
        for k in range(a+1, b): tags[k] = f"I-{t}"
    while i < n:
        w = tokens[i]
        # dd / mm / yyyy  (5 tokens after split_tags)
        if (i+4 < n and RX["DATE_D"].match(w) and RX["SEP"].match(tokens[i+1])
                and RX["DATE_D"].match(tokens[i+2]) and RX["SEP"].match(tokens[i+3])
                and (RX["DATE_Y"].match(tokens[i+4]) or RX["DATE_D"].match(tokens[i+4]))):
            put(i, i+5, "DATE"); i += 5; continue
        # ( DD ) 99999 - 9999
        if (i+5 < n and w == "(" and RX["DDD"].match(tokens[i+1]) and tokens[i+2] == ")"
                and RX["PHONE_P"].match(tokens[i+3]) and tokens[i+4] == "-"
                and RX["PHONE_S"].match(tokens[i+5])):
            put(i+1, i+2, "PHONE"); put(i+3, i+6, "PHONE"); i += 6; continue
        if (i+2 < n and RX["PHONE_P"].match(w) and tokens[i+1] == "-"
                and RX["PHONE_S"].match(tokens[i+2])):
            put(i, i+3, "PHONE"); i += 3; continue
        # user @ domain.tld
        if (i+2 < n and RX["EMAIL_U"].match(w) and tokens[i+1] == "@"
                and RX["EMAIL_D"].match(tokens[i+2])):
            put(i, i+3, "EMAIL"); i += 3; continue
        if RX["ZIP"].match(w):  put(i, i+1, "ZIP");   i += 1; continue
        if RX["CPF"].match(w):  put(i, i+1, "IDNUM"); i += 1; continue
        if RX["IDNUM"].match(w): put(i, i+1, "IDNUM"); i += 1; continue
        # age: number immediately followed by a temporal cue
        if (RX["DATE_D"].match(w) and i+1 < n and tokens[i+1].lower() in AGE_CUE):
            put(i, i+1, "AGE"); i += 1; continue
        i += 1
    return tags

_demo = CORPUS["test"][0]["tokens"]
print("demo:", list(zip(_demo, regex_tag(_demo)))[:14])

---
## 5. Teacher — constrained decoding and real soft labels



In [ ]:
from transformers import AutoTokenizer, AutoModelForCausalLM

PREFIXES  = ["O", "B", "I"]      # the model emits these natively
NONE_CHAR = "-"                  # type slot filler when the prefix is O

def build_label_alphabet(tok) -> Tuple[Dict[str, int], Dict[str, str]]:
    """BIO prefix ids plus one single-token character per entity type.

    History, because each failure taught something:

      v1: 39 arbitrary characters. 'O' landed on I-HEALTH_PLAN, so 98.5% of the
          output decoded to that rare class -- the model meant *outside*.
      v2: 'O' reserved for O. Agreement hit 86.5%, but that is the majority-class
          baseline: the teacher predicted O for 99.6% of tokens.
      v3: split into prefix (3-way) then type (19-way). The type stage worked --
          DOCTOR, PATIENT, PHONE all appeared -- but the prefix stage still said
          O for 98.8% of tokens. Entity recall 1.4%.

    v4 keeps the two stages and adds a forced echo (see generate_soft_labels):
    the word is emitted immediately before its tag, so the decision is local and
    the model never has to count its own output to know where it is.
    """
    # Qwen uses GPT-style BPE, where a token carries its preceding space. If the
    # echo ends with a space and we then force a bare "O", the model is in a
    # state it essentially never sees in training and the O/B/I logits stop
    # meaning anything. Prefer space-prefixed tag tokens when they exist.
    global TAG_LEADING_SPACE
    TAG_LEADING_SPACE = all(
        len(tok.encode(" " + x, add_special_tokens=False)) == 1 for x in PREFIXES)
    _sp = " " if TAG_LEADING_SPACE else ""
    print(f"tag tokens use leading space: {TAG_LEADING_SPACE}")

    pref_ids = {}
    for pfx in PREFIXES:
        e = tok.encode(_sp + pfx, add_special_tokens=False)
        assert len(e) == 1, f"prefix {pfx!r} is not a single token"
        pref_ids[pfx] = e[0]
    assert len(set(pref_ids.values())) == 3, "B/I/O token ids collide"

    types = SUBCATEGORIES if CFG.granularity == "sub" else COARSE
    pool = "0123456789acdefghjklmnpqrstuvwxyzACDEFGHJKLMNPQRSTUVWXYZ!#$%&*+;=?^~"
    type_chars, used_ids = {}, set()
    e = tok.encode(NONE_CHAR, add_special_tokens=False)
    assert len(e) == 1, "NONE_CHAR is not a single token"
    type_chars["_NONE"] = NONE_CHAR; used_ids.add(e[0])
    for t in types:
        for ch in pool:
            if ch in type_chars.values():
                continue
            e = tok.encode(ch, add_special_tokens=False)
            if len(e) == 1 and e[0] not in used_ids:
                type_chars[t] = ch; used_ids.add(e[0]); break
        else:
            raise RuntimeError(f"no single-token character left for type {t}")
    assert len(set(type_chars.values())) == len(types) + 1, "type characters collide"
    return pref_ids, type_chars

FEWSHOT: List[Tuple[List[str], List[str]]] = []      # filled from the training set

def make_fewshot(n_examples=2, max_words=35):
    """Draw real annotated windows from training data for the prompt.

    The earlier hand-written example ("Paciente Maria Silva, 45 anos") looks
    nothing like this corpus, which is full of section rules, ALL-CAPS headers
    and spaCy-split punctuation. Showing the model what the actual input looks
    like, with its gold tags, is the cheapest remaining lever.
    """
    picked = []
    for c in CORPUS["train"]:
        tags = c["tags"][:max_words]
        if sum(t != "O" for t in tags) >= 4:
            picked.append((c["tokens"][:max_words], tags))
        if len(picked) >= n_examples:
            break
    return picked

def build_prompt(tok, words: List[str], type_chars: Dict[str, str]) -> str:
    """ChatML prompt. The note is shown as running text first, then word by word.

    The full prompt format is serialised into manifest.json (section 15), so the
    exact text sent to the teacher is recoverable from the released artefacts.
    """
    types = SUBCATEGORIES if CFG.granularity == "sub" else COARSE
    legend = "\n".join(f"{type_chars[t]} = {t}" for t in types)
    def render(ws, tgs):
        out = []
        for i, (w, t) in enumerate(zip(ws, tgs)):
            pfx = t[0] if t != "O" else "O"
            typ = type_chars.get(t[2:], type_chars["_NONE"]) if t != "O" else type_chars["_NONE"]
            out.append(f"{i+1}. {w} {pfx}{typ}")
        return "\n".join(out)

    if FEWSHOT:
        example = "\n\n".join(f"EXEMPLO {k+1}:\n" + render(ws, tgs)
                              for k, (ws, tgs) in enumerate(FEWSHOT))
    else:
        ex_w = ["Paciente","Maria","Silva",",","45","anos",",","atendida","por","Dr.","Souza"]
        ex_p = ["O","B","I","O","B","O","O","O","O","O","B"]
        ex_t = ["_NONE","PATIENT","PATIENT","_NONE","AGE","_NONE","_NONE","_NONE",
                "_NONE","_NONE","DOCTOR"]
        example = "EXEMPLO:\n" + "\n".join(
            f"{i+1}. {w} {pp}{type_chars[tt]}"
            for i, (w, pp, tt) in enumerate(zip(ex_w, ex_p, ex_t)))
    sys_msg = (
        "Você identifica informações pessoais (PHI) em prontuários médicos "
        "brasileiros, para fins de anonimização.\n\n"
        "Para CADA palavra, escreva o número, a palavra e DOIS caracteres:\n"
        "1) prefixo BIO — B inicia uma entidade, I continua a entidade anterior, "
        "O não é informação pessoal;\n"
        f"2) código do tipo (ou {NONE_CHAR} quando o prefixo for O).\n\n"
        f"CÓDIGOS DE TIPO:\n{legend}\n{NONE_CHAR} = não aplicável\n\n"
        f"{example}\n\n"
        "SÃO informação pessoal: nomes de pacientes e de profissionais, datas, "
        "idades, telefones, e-mails, endereços, cidades, estados, hospitais, "
        "convênios, números de documento e de prontuário.\n"
        "NÃO são: termos clínicos, diagnósticos, medicamentos, sintomas, exames, "
        "condutas e pontuação.\n"
        "Um prontuário típico contém muitas entidades — não marque tudo como O.")
    user_msg = ("Texto do prontuário:\n" + " ".join(words)
                + f"\n\nAgora rotule as {len(words)} palavras, uma por linha:")
    return tok.apply_chat_template(
        [{"role": "system", "content": sys_msg}, {"role": "user", "content": user_msg}],
        tokenize=False, add_generation_prompt=True)

def build_template(tok, words: List[str], pref_order, type_ids, nl_id):
    """Decoding template: forced echo tokens with two free slots per word.

    Each word expands to  '<n>. <word> ' + [prefix] + [type] + '\n'.
    Everything except the two bracketed slots is teacher-forced, so the model
    reads the word immediately before tagging it, and step -> word alignment is
    exact by construction. No parsing of the output is ever required.

    Returns (template, free_slots) where template[s] is (allowed_ids, kind) and
    free_slots[w] = (prefix_step, type_step).
    """
    tpl, free = [], []
    tail = "" if TAG_LEADING_SPACE else " "
    for i, w in enumerate(words):
        for tid in tok.encode(f"{i+1}. {w}{tail}", add_special_tokens=False):
            tpl.append(([tid], "force"))
        free.append((len(tpl), len(tpl) + 1))
        tpl.append((pref_order, "prefix"))
        tpl.append((type_ids,  "type"))
        tpl.append(([nl_id],   "force"))
    return tpl, free

print("teacher prompt builder defined")

In [ ]:
def load_teacher():
    tok = AutoTokenizer.from_pretrained(CFG.teacher, padding_side="left")
    model = AutoModelForCausalLM.from_pretrained(
        CFG.teacher, torch_dtype=getattr(torch, CFG.teacher_dtype), device_map="auto")
    model.eval()
    n_par = sum(p.numel() for p in model.parameters())
    size_mb = sum(p.numel()*p.element_size() for p in model.parameters())/1e6
    print(f"teacher: {n_par/1e9:.2f}B params, {size_mb:.0f} MB in {CFG.teacher_dtype}")
    return tok, model, {"params_M": round(n_par/1e6), "size_MB": round(size_mb)}

@torch.no_grad()
def generate_soft_labels(tok, model, chunks, debug=0):
    """Forced-echo constrained decoding -> per-word distribution over BIO labels.

    The generation template per word is

        '<n>. <word> '   forced (teacher-forced echo)
        [prefix]         free, constrained to {O, B, I}
        [type]           free, constrained to the type characters
        '\n'             forced

    so the model sees the word immediately before deciding, and the mapping from
    decoding step to word is exact. This replaces free-form TOKEN<TAB>TAG output,
    whose step-to-word alignment is not guaranteed and must be recovered by
    parsing.

    The cached distribution is the joint

        P(O)   = p_prefix[O]
        P(B-X) = p_prefix[B] * p_type[X]
        P(I-X) = p_prefix[I] * p_type[X]

    with p_type renormalised over real types (the "not applicable" slot is
    dropped), stored as log-probabilities. The KD loss applies softmax(z / T),
    and softmax(log p / T) is exactly the tempered distribution -- temperature is
    therefore applied once and only once.
    """
    pref_ids, type_chars = build_label_alphabet(tok)
    types = SUBCATEGORIES if CFG.granularity == "sub" else COARSE
    pref_order = [pref_ids[x] for x in PREFIXES]                    # O, B, I
    type_keys  = ["_NONE"] + types
    type_ids   = [tok.encode(type_chars[k], add_special_tokens=False)[0] for k in type_keys]
    nl = tok.encode("\n", add_special_tokens=False)
    assert len(nl) == 1, "newline is not a single token in this vocabulary"
    nl_id, eos_id = nl[0], (tok.eos_token_id or tok.pad_token_id)

    idx_O = LABEL2ID["O"]
    idx_B = [LABEL2ID[f"B-{t}"] for t in types]
    idx_I = [LABEL2ID[f"I-{t}"] for t in types]

    cache, stats = {}, {"positions": 0, "fallback": 0, "chunks": 0, "truncated_chunks": 0}
    t0 = time.time()
    for start in range(0, len(chunks), CFG.teacher_batch):
        batch = chunks[start:start + CFG.teacher_batch]
        words = [c["tokens"][:CFG.teacher_max_words] for c in batch]
        prompts = [build_prompt(tok, w, type_chars) for w in words]
        enc = tok(prompts, return_tensors="pt", padding=True).to(model.device)
        plen = enc.input_ids.shape[1]

        built = [build_template(tok, w, pref_order, type_ids, nl_id) for w in words]
        tmpls = [t for t, _ in built]
        frees = [f for _, f in built]

        def allowed(batch_id, input_ids):
            s = input_ids.shape[0] - plen
            tpl = tmpls[batch_id]
            return tpl[s][0] if s < len(tpl) else [eos_id]

        out = model.generate(**enc, max_new_tokens=max(len(t) for t in tmpls) + 1,
                             do_sample=False, prefix_allowed_tokens_fn=allowed,
                             return_dict_in_generate=True, output_scores=True,
                             pad_token_id=tok.pad_token_id or tok.eos_token_id)
        n_steps = len(out.scores)

        for b, chunk in enumerate(batch):
            nw = len(chunk["tokens"])
            arr = np.full((nw, N_LABELS), -20.0, dtype=np.float16)
            done = 0
            for w, (sp, st) in enumerate(frees[b]):
                if w >= nw or st >= n_steps:
                    break
                pp = torch.softmax(out.scores[sp][b, pref_order].float(), -1).cpu().numpy()
                pt = torch.softmax(out.scores[st][b, type_ids].float(), -1).cpu().numpy()
                pt_real = pt[1:]                                   # drop the _NONE slot
                pt_real = pt_real / max(pt_real.sum(), 1e-9)
                joint = np.full(N_LABELS, 1e-9, dtype=np.float32)
                joint[idx_O] = pp[0]
                joint[idx_B] = pp[1] * pt_real
                joint[idx_I] = pp[2] * pt_real
                arr[w] = np.log(joint).astype(np.float16)
                done = w + 1
            for w in range(done, nw):                              # generation ran short
                stats["fallback"] += 1
                arr[w, chunk["label_ids"][w]] = 0.0                # log(1) on the gold label
            stats["positions"] += nw
            if nw > CFG.teacher_max_words:
                stats["truncated_chunks"] += 1
            cache[chunk["id"]] = arr

        if debug and start == 0:
            txt = tok.decode(out.sequences[0][plen:], skip_special_tokens=True)
            print("--- raw teacher output (first example) ---")
            print("\n".join(txt.splitlines()[:debug]))
            print("--- end ---\n")

        stats["chunks"] += len(batch)
        if (start // CFG.teacher_batch) % 20 == 0:
            el = time.time() - t0; d = start + len(batch)
            print(f"  {d}/{len(chunks)}  {el/60:.1f} min  "
                  f"ETA {el/max(d,1)*(len(chunks)-d)/60:.0f} min  "
                  f"fallback {100*stats['fallback']/max(stats['positions'],1):.2f}%")

    stats["fallback_rate"] = stats["fallback"] / max(stats["positions"], 1)
    stats["minutes"] = round((time.time() - t0) / 60, 1)
    return cache, stats

SOFT_PATH  = f"{CFG.out_dir}/soft_labels.npz"
STATS_PATH = f"{CFG.out_dir}/teacher_stats.json"
print("soft label generator defined")

In [ ]:
# --- Run the teacher (or load the cache). No simulated branch exists. ---
if Path(SOFT_PATH).exists() and Path(STATS_PATH).exists():
    _z = np.load(SOFT_PATH)
    SOFT = {k: _z[k] for k in _z.files}
    TEACHER_STATS = json.loads(Path(STATS_PATH).read_text())
    TEACHER_META  = TEACHER_STATS.get("meta", {})
    print(f"loaded cache: {len(SOFT)} chunks")
else:
    t_tok, t_model, TEACHER_META = load_teacher()
    FEWSHOT = make_fewshot(n_examples=2, max_words=35)
    print(f"few-shot: {len(FEWSHOT)} real annotated windows from the training set")

    # Cheap probe first: one batch, with the raw output printed. Prompt design is
    # the dominant failure mode at this stage, so inspect what the model actually
    # writes before committing GPU hours to the full run.
    _cap = CFG.teacher_max_words
    CFG.teacher_max_words = min(60, _cap)          # keep the probe to seconds
    _pchunks = [dict(c, tokens=c["tokens"][:60], tags=c["tags"][:60],
                     label_ids=c["label_ids"][:60]) for c in CORPUS["train"][:2]]
    _probe, _ = generate_soft_labels(t_tok, t_model, _pchunks, debug=25)
    CFG.teacher_max_words = _cap
    _probe_pred = sum(int(ID2LABEL[int(i)] != "O")
                      for k in _probe for i in _probe[k].argmax(-1))
    _probe_gold = sum(int(t != "O") for c in _pchunks for t in c["tags"])
    print(f"PROBE: teacher marks {_probe_pred} entity tokens, gold has {_probe_gold}")
    if _probe_pred < 0.2 * _probe_gold:
        raise AssertionError(
            "Probe failed: the teacher is not marking entities. Inspect the raw "
            "output above before spending GPU hours. Options: more few-shot "
            "examples, CFG.granularity='coarse', or accept a weak teacher and "
            "report it as a finding (set CFG.allow_weak_teacher=True).")

    SOFT, TEACHER_STATS = generate_soft_labels(t_tok, t_model, CORPUS["train"])
    TEACHER_STATS["meta"] = TEACHER_META
    np.savez_compressed(SOFT_PATH, **SOFT)
    Path(STATS_PATH).write_text(json.dumps(TEACHER_STATS, indent=2))
    del t_model; torch.cuda.empty_cache()
    print("cache written -- if the session dies, re-running this cell reloads it "
          "in seconds without touching the GPU")

missing = {c["id"] for c in CORPUS["train"]} - set(SOFT)
if missing: raise AssertionError(f"{len(missing)} training chunks lack soft labels")

# Diagnostic. Overall token agreement is worthless here: 86.6% of gold is O, so
# a teacher that answers "not PHI" everywhere scores 86.6% while carrying zero
# signal -- which is exactly what the previous prompt did. What matters is
# behaviour on the positions that are actually entities.
_tp = _fp = _fn = _n = _agree = 0
_pred, _gold = Counter(), Counter()
for _c in CORPUS["train"]:
    if _c["id"] not in SOFT: continue
    _pa = SOFT[_c["id"]].argmax(-1)
    for _i, _g in enumerate(_c["label_ids"][:len(_pa)]):
        _p = int(_pa[_i]); _n += 1; _agree += int(_p == _g)
        _pred[ID2LABEL[_p]] += 1; _gold[ID2LABEL[_g]] += 1
        _pe, _ge = ID2LABEL[_p] != "O", ID2LABEL[_g] != "O"
        _tp += int(_pe and _ge and _p == _g); _fp += int(_pe and not _ge)
        _fn += int(_ge and not _pe)
_ent_gold = sum(v for k, v in _gold.items() if k != "O")
_ent_pred = sum(v for k, v in _pred.items() if k != "O")
_rec = _tp / max(_ent_gold, 1)
print(f"positions: {_n:,}   overall agreement: {100*_agree/max(_n,1):.1f}% "
      f"(majority-class baseline: {100*_gold['O']/max(_n,1):.1f}%)")
print(f"gold entity tokens: {_ent_gold:,}   teacher entity tokens: {_ent_pred:,}")
print(f"teacher entity recall (exact label): {100*_rec:.1f}%   FP: {_fp:,}  FN: {_fn:,}")
print("teacher top labels:", _pred.most_common(5))
if _ent_pred < 0.2 * _ent_gold and not CFG.allow_weak_teacher:
    raise AssertionError(
        f"Teacher marks only {_ent_pred:,} entity tokens against {_ent_gold:,} in gold "
        "-- it is answering 'not PHI' almost everywhere and the soft labels carry no "
        "usable signal.\n\nOptions:\n"
        "  1. CFG.granularity = 'coarse'  (8 types instead of 19)\n"
        "  2. more/longer few-shot examples in make_fewshot()\n"
        "  3. QLoRA-tune the teacher before generation\n"
        "  4. CFG.allow_weak_teacher = True -- proceed and report the weak teacher\n"
        "     as the finding, which is a legitimate and publishable outcome\n\n"
        "Delete the cache to regenerate.")
if _rec < 0.30:
    print("[!] Entity recall below 30%. Distillation will transfer little; consider "
          "more few-shot examples, coarse granularity, or a fine-tuned teacher.")

print(f"\nFALLBACK RATE: {100*TEACHER_STATS.get('fallback_rate', 0):.2f}% of "
      f"{TEACHER_STATS.get('positions', 0):,} positions")
print(f"truncated chunks (> {CFG.teacher_max_words} words): "
      f"{TEACHER_STATS.get('truncated_chunks', 0)}")
if TEACHER_STATS.get("fallback_rate", 0) > 0.05:
    print("[!] Above 5% -- a meaningful share of supervision is hard-label "
          "rather than distilled. Report this rate with the results.")

---
## 6. Student dataset, alignment and KD loss



In [ ]:
from torch.utils.data import Dataset, DataLoader
from transformers import (AutoModelForTokenClassification, TrainingArguments, Trainer,
                          default_data_collator)

student_tok = AutoTokenizer.from_pretrained(CFG.student)

class NERSet(Dataset):
    def __init__(self, chunks, soft: Optional[dict] = None):
        self.chunks, self.soft = chunks, soft
    def __len__(self): return len(self.chunks)
    def __getitem__(self, i):
        c = self.chunks[i]
        enc = student_tok(c["tokens"], is_split_into_words=True, truncation=True,
                          max_length=CFG.max_length, padding="max_length",
                          return_tensors="pt")
        wids = enc.word_ids(0)
        labels = torch.full((CFG.max_length,), -100, dtype=torch.long)
        soft = torch.zeros(CFG.max_length, N_LABELS) if self.soft is not None else None
        prev = None
        for pos, wid in enumerate(wids):
            if wid is None or wid == prev: prev = wid; continue
            labels[pos] = c["label_ids"][wid]
            if soft is not None:
                soft[pos] = torch.from_numpy(self.soft[c["id"]][wid].astype(np.float32))
            prev = wid
        item = {"input_ids": enc.input_ids[0], "attention_mask": enc.attention_mask[0],
                "labels": labels}
        if soft is not None: item["soft_logits"] = soft
        return item

class KDTrainer(Trainer):
    """L = a*CE(hard) + (1-a)*T^2*KL(teacher_T || student_T), masked at -100."""
    def __init__(self, *a, kd_T=4.0, kd_alpha=0.5, **kw):
        super().__init__(*a, **kw); self.kd_T, self.kd_alpha = kd_T, kd_alpha
    def compute_loss(self, model, inputs, return_outputs=False, **kw):
        soft = inputs.pop("soft_logits", None)
        labels = inputs["labels"]
        out = model(input_ids=inputs["input_ids"], attention_mask=inputs["attention_mask"])
        logits = out.logits
        ce = F.cross_entropy(logits.view(-1, N_LABELS), labels.view(-1), ignore_index=-100)
        if soft is None:
            return (ce, out) if return_outputs else ce
        mask = labels.view(-1) != -100
        T = self.kd_T
        s = F.log_softmax(logits.view(-1, N_LABELS)[mask] / T, dim=-1)
        t = F.softmax(soft.view(-1, N_LABELS)[mask] / T, dim=-1)
        kl = F.kl_div(s, t, reduction="batchmean") * (T ** 2)
        loss = self.kd_alpha * ce + (1 - self.kd_alpha) * kl
        return (loss, out) if return_outputs else loss

print("dataset + KD trainer defined")

---
## 7. Entity-level evaluation



In [ ]:
from seqeval.metrics import classification_report as seq_report
from seqeval.scheme import IOB2

@torch.no_grad()
def predict(model, chunks, device=None, batch_size=32):
    """Return (y_true, y_pred) as lists of per-sequence tag lists."""
    device = device or ("cuda" if torch.cuda.is_available() else "cpu")
    model = model.to(device).eval()
    ds = NERSet(chunks)
    yt, yp = [], []
    for i in range(0, len(ds), batch_size):
        items = [ds[j] for j in range(i, min(i+batch_size, len(ds)))]
        ii = torch.stack([x["input_ids"] for x in items]).to(device)
        am = torch.stack([x["attention_mask"] for x in items]).to(device)
        lb = torch.stack([x["labels"] for x in items])
        pr = model(input_ids=ii, attention_mask=am).logits.argmax(-1).cpu()
        for b in range(len(items)):
            m = lb[b] != -100
            yt.append([ID2LABEL[int(v)] for v in lb[b][m]])
            yp.append([ID2LABEL[int(v)] for v in pr[b][m]])
    return yt, yp

def entity_metrics(y_true, y_pred) -> dict:
    rep = seq_report(y_true, y_pred, output_dict=True, mode="strict",
                     scheme=IOB2, zero_division=0)
    return {"macro_f1": rep["macro avg"]["f1-score"],
            "micro_f1": rep["micro avg"]["f1-score"],
            "macro_precision": rep["macro avg"]["precision"],
            "macro_recall": rep["macro avg"]["recall"],
            "micro_precision": rep["micro avg"]["precision"],
            "micro_recall": rep["micro avg"]["recall"],
            "per_type": {k: v for k, v in rep.items() if not k.endswith("avg")}}

def kept_words(chunk) -> int:
    """How many words survive WordPiece truncation at max_length.

    predict() only returns positions that carry a label, so a window whose
    sub-word expansion exceeds max_length is evaluated on a PREFIX of its words.
    The rule baseline must be scored on exactly the same prefix, otherwise the
    two systems are measured on different label sequences and the headline
    comparison is invalid.
    """
    enc = student_tok(chunk["tokens"], is_split_into_words=True,
                      truncation=True, max_length=CFG.max_length)
    wids = [w for w in enc.word_ids() if w is not None]
    return (max(wids) + 1) if wids else 0

def rule_predictions(chunks):
    yt, yp = [], []
    for c in chunks:
        k = kept_words(c)
        yt.append(c["tags"][:k])
        yp.append(regex_tag(c["tokens"])[:k])
    return yt, yp

# Report truncation: it belongs in the methods section, and a high rate means
# the window size and max_length are mismatched.
_lost = [(len(c["tokens"]) - kept_words(c)) for c in CORPUS_EVAL["test"]]
_n_trunc = sum(1 for x in _lost if x > 0)
print(f"truncation: {_n_trunc}/{len(_lost)} eval windows lose words "
      f"({sum(_lost)} words total, max {max(_lost) if _lost else 0} in one window)")
if sum(_lost) > 0.01 * sum(len(c["tokens"]) for c in CORPUS_EVAL["test"]):
    print("[!] Over 1% of words are truncated. Lower CFG.eval_window or raise "
          "CFG.max_length -- and report this in the methods section.")

_yt, _yp = rule_predictions(CORPUS_EVAL["test"])
RULE_METRICS = entity_metrics(_yt, _yp)
print(f"regex baseline  macro F1={RULE_METRICS['macro_f1']:.4f}  "
      f"micro F1={RULE_METRICS['micro_f1']:.4f}  recall={RULE_METRICS['micro_recall']:.4f}")
print("\nper-type F1 (rules only, structured categories):")
for t, v in sorted(RULE_METRICS["per_type"].items(), key=lambda kv: -kv[1]["f1-score"])[:8]:
    print(f"  {t:16s} P={v['precision']:.3f} R={v['recall']:.3f} F1={v['f1-score']:.3f} n={v['support']}")

---
## 8. Paired hyperparameter grid



In [ ]:
def train_one(train_chunks, val_chunks, seed, lr, epochs, soft=None,
              kd_T=None, kd_alpha=None, tag="run"):
    set_seed(seed)
    model = AutoModelForTokenClassification.from_pretrained(
        CFG.student, num_labels=N_LABELS, id2label=ID2LABEL, label2id=LABEL2ID)
    args = TrainingArguments(
        output_dir=f"{CFG.out_dir}/tmp_{tag}", report_to="none",
        learning_rate=lr, num_train_epochs=epochs,
        per_device_train_batch_size=CFG.batch_size,
        per_device_eval_batch_size=32, warmup_ratio=CFG.warmup_ratio,
        weight_decay=CFG.weight_decay, lr_scheduler_type="cosine",
        seed=seed, data_seed=seed, save_strategy="no", logging_strategy="no",
        bf16=torch.cuda.is_available(), disable_tqdm=True,

        remove_unused_columns=False)
    tr = KDTrainer(model=model, args=args,
                   train_dataset=NERSet(train_chunks, soft),
                   kd_T=kd_T or CFG.kd_T, kd_alpha=kd_alpha if kd_alpha is not None else CFG.kd_alpha,
                   data_collator=default_data_collator)   # NERSet already pads to max_length;
                                                          # the token-classification collator
                                                          # cannot pad the soft_logits tensor
    tr.train()
    yt, yp = predict(model, val_chunks)
    return model, entity_metrics(yt, yp)

def train_and_discard(*a, **kw):
    """Grid runs only need the metrics. Free the weights or the grid OOMs."""
    model, met = train_one(*a, **kw)
    del model
    gc.collect(); torch.cuda.empty_cache()
    return met

def assert_kd_active():
    """Fail loudly if the KD term is not reaching the loss.

    A silently-dropped soft_logits key makes KD numerically identical to the
    baseline while every log looks healthy. Check it once, up front.
    """
    set_seed(0)
    m = AutoModelForTokenClassification.from_pretrained(
        CFG.student, num_labels=N_LABELS, id2label=ID2LABEL, label2id=LABEL2ID)
    chunks = CORPUS["train"][:2]
    batch_hard = default_data_collator([NERSet(chunks)[i] for i in range(len(chunks))])
    batch_soft = default_data_collator([NERSet(chunks, SOFT)[i] for i in range(len(chunks))])
    assert "soft_logits" in batch_soft, "collator dropped soft_logits"
    args = TrainingArguments(output_dir=f"{CFG.out_dir}/tmp_probe", report_to="none",
                             remove_unused_columns=False, disable_tqdm=True)
    tr = KDTrainer(model=m, args=args, kd_T=CFG.kd_T, kd_alpha=CFG.kd_alpha,
                   data_collator=default_data_collator)
    # Trainer moves the model to args.device on construction, so the probe
    # batches must follow it or this dies on a CPU/CUDA mismatch.
    dev = next(tr.model.parameters()).device
    to_dev = lambda b: {k: v.to(dev) for k, v in b.items()}
    l_hard = float(tr.compute_loss(tr.model, to_dev(dict(batch_hard))))
    l_soft = float(tr.compute_loss(tr.model, to_dev(dict(batch_soft))))
    print(f"loss without soft labels = {l_hard:.4f}")
    print(f"loss with    soft labels = {l_soft:.4f}")
    if abs(l_hard - l_soft) < 1e-6:
        raise AssertionError(
            "KD loss equals the hard-label loss -- the distillation term is not "
            "being applied. Check remove_unused_columns=False and that SOFT "
            "covers every training chunk id.")
    print("KD term is active\n")
    del m; torch.cuda.empty_cache()

assert_kd_active()

LR_GRID     = [2e-5, 3e-5, 5e-5]
EPOCH_GRID  = [3, 5]
KD_GRID     = [(2.0,0.3),(2.0,0.5),(4.0,0.3),(4.0,0.5),(4.0,0.7),(6.0,0.5),(8.0,0.5)]
if CFG.SMOKE:
    LR_GRID, EPOCH_GRID, KD_GRID = [3e-5], [1], [(4.0,0.5),(4.0,0.7)]

GRID_SEED = CFG.seeds[0]
rows = []


for cond, soft in (("baseline", None), ("kd", SOFT)):
    for lr in LR_GRID:
        for ep in EPOCH_GRID:
            m = train_and_discard(CORPUS["train"], CORPUS_EVAL["validation"],
                                  GRID_SEED, lr, ep, soft=soft,
                                  tag=f"{cond}_{lr}_{ep}")
            rows.append({"model": cond, "lr": lr, "epochs": ep,
                         "T": CFG.kd_T if soft is not None else None,
                         "alpha": CFG.kd_alpha if soft is not None else None,
                         "val_macro_f1": m["macro_f1"], "val_micro_f1": m["micro_f1"]})
            print(f"{cond:8s} lr={lr} ep={ep}  val macro F1={m['macro_f1']:.4f}")

best_base = max([r for r in rows if r["model"] == "baseline"],
                key=lambda r: r["val_macro_f1"])
best_kd_hp = max([r for r in rows if r["model"] == "kd"],
                 key=lambda r: r["val_macro_f1"])
print(f"\n-> best baseline: lr={best_base['lr']} epochs={best_base['epochs']}")
print(f"-> best KD optimiser: lr={best_kd_hp['lr']} epochs={best_kd_hp['epochs']}")

# Distillation hyperparameters, at the KD arm's own best optimiser settings.
for T, a in KD_GRID:
    m = train_and_discard(CORPUS["train"], CORPUS_EVAL["validation"], GRID_SEED,
                          best_kd_hp["lr"], best_kd_hp["epochs"], soft=SOFT,
                          kd_T=T, kd_alpha=a, tag=f"kdTa_{T}_{a}")
    rows.append({"model": "kd", "lr": best_kd_hp["lr"], "epochs": best_kd_hp["epochs"],
                 "T": T, "alpha": a,
                 "val_macro_f1": m["macro_f1"], "val_micro_f1": m["micro_f1"]})
    print(f"KD T={T} a={a}  val macro F1={m['macro_f1']:.4f}")

GRID = pd.DataFrame(rows); GRID.to_csv(f"{CFG.out_dir}/grid.csv", index=False)
best_kd = max([r for r in rows if r["model"] == "kd"], key=lambda r: r["val_macro_f1"])
print(f"\n-> best KD: lr={best_kd['lr']} epochs={best_kd['epochs']} "
      f"T={best_kd['T']} alpha={best_kd['alpha']}")
print("The reported configuration is whichever one this grid selects.")

---
## 9. Final multi-seed training



In [ ]:
FINAL = {"baseline": [], "kd": []}
MODELS = {}

for seed in CFG.seeds:
    mb, met_b = train_one(CORPUS["train"], CORPUS_EVAL["test"], seed,
                          best_base["lr"], best_base["epochs"], tag=f"fb_{seed}")
    FINAL["baseline"].append({"seed": seed, **{k: v for k, v in met_b.items() if k != "per_type"}})
    mk, met_k = train_one(CORPUS["train"], CORPUS_EVAL["test"], seed,
                          best_kd["lr"], best_kd["epochs"], soft=SOFT,
                          kd_T=best_kd["T"], kd_alpha=best_kd["alpha"], tag=f"fk_{seed}")
    FINAL["kd"].append({"seed": seed, **{k: v for k, v in met_k.items() if k != "per_type"}})
    print(f"seed {seed}: baseline {met_b['macro_f1']:.4f} | KD {met_k['macro_f1']:.4f} "
          f"| delta {100*(met_k['macro_f1']-met_b['macro_f1']):+.2f} pp")
    if seed == CFG.seeds[0]:
        MODELS["baseline"], MODELS["kd"] = mb, mk
        PER_TYPE = {"baseline": met_b["per_type"], "kd": met_k["per_type"]}

for cond in FINAL:
    f = [r["macro_f1"] for r in FINAL[cond]]
    print(f"{cond:9s} macro F1 = {np.mean(f):.4f} +/- {np.std(f, ddof=1) if len(f)>1 else 0:.4f} (n={len(f)})")

---
## 10. INT8 quantisation and measured latency



In [ ]:
try:                                        # torch >= 1.13
    from torch.ao.quantization import quantize_dynamic
except ImportError:                         # older releases
    from torch.quantization import quantize_dynamic

def quantize_int8(model):
    return quantize_dynamic(model.cpu().eval(), {nn.Linear}, dtype=torch.qint8)

def model_size_mb(model) -> float:
    p = f"{CFG.out_dir}/_tmp.pt"; torch.save(model.state_dict(), p)
    mb = os.path.getsize(p)/1e6; os.remove(p); return round(mb, 1)

@torch.no_grad()
def benchmark_cpu(model, chunks, batch_size=16, n_batches=30, warmup=5, threads=None):
    """Measure real CPU latency. Records the ACTUAL tensor shape measured."""
    threads = threads or min(8, os.cpu_count() or 1)
    torch.set_num_threads(threads)
    model = model.cpu().eval()
    ds = NERSet(chunks); lat, shapes = [], []
    for i in range(n_batches + warmup):
        s = (i * batch_size) % max(len(ds) - batch_size, 1)
        items = [ds[j] for j in range(s, min(s + batch_size, len(ds)))]
        ii = torch.stack([x["input_ids"] for x in items])
        am = torch.stack([x["attention_mask"] for x in items])
        t0 = time.perf_counter(); model(input_ids=ii, attention_mask=am)
        dt = (time.perf_counter() - t0) * 1000
        if i >= warmup: lat.append(dt); shapes.append(tuple(ii.shape))
    mean = float(np.mean(lat)); shape = shapes[0]
    tok_per_batch = shape[0] * shape[1]
    return {"latency_ms_mean": round(mean, 1), "latency_ms_std": round(float(np.std(lat)), 1),
            "batch_shape": list(shape), "tokens_per_batch": tok_per_batch,
            "tokens_per_second": round(tok_per_batch / (mean/1000)),
            "threads": threads, "cpu": ENV.get("cpu_model")}

kd_fp32 = MODELS["kd"]
kd_int8 = quantize_int8(kd_fp32)

BENCH = {}
for name, mdl in (("baseline", MODELS["baseline"]), ("kd_fp32", kd_fp32), ("kd_int8", kd_int8)):
    b = benchmark_cpu(mdl, CORPUS_EVAL["test"][:200])
    b["size_MB"] = model_size_mb(mdl)
    BENCH[name] = b
    print(f"{name:10s} {b['latency_ms_mean']:7.1f} ms/batch  shape={b['batch_shape']}  "
          f"{b['tokens_per_second']:>8,} tok/s  {b['size_MB']:6.1f} MB")

yt_q, yp_q = predict(kd_int8, CORPUS_EVAL["test"], device="cpu")
INT8_METRICS = entity_metrics(yt_q, yp_q)
print(f"\nINT8 macro F1 = {INT8_METRICS['macro_f1']:.4f} "
      f"(FP32 {FINAL['kd'][0]['macro_f1']:.4f})")
print(f"size {BENCH['kd_fp32']['size_MB']:.0f} -> {BENCH['kd_int8']['size_MB']:.0f} MB "
      f"({100*(1-BENCH['kd_int8']['size_MB']/BENCH['kd_fp32']['size_MB']):.1f}% reduction)")

In [ ]:
yt_t, yp_base = predict(MODELS["baseline"], CORPUS_EVAL["test"])
_,    yp_kd   = predict(MODELS["kd"],       CORPUS_EVAL["test"])
print(f"predictions: {len(yt_t)} sequences")

---
## 11. Statistical inference aligned with the endpoint



In [ ]:
from scipy.stats import wilcoxon

def span_set(seq):
    out, cur = [], None
    for i, t in enumerate(seq):
        if t.startswith("B-"):
            if cur: out.append(tuple(cur))
            cur = [t[2:], i, i+1]
        elif t.startswith("I-") and cur and cur[0] == t[2:]:
            cur[2] = i + 1
        else:
            if cur: out.append(tuple(cur)); cur = None
    if cur: out.append(tuple(cur))
    return set(out)

def count_matrix(y_true, y_pred, types):
    # Per-document TP/FP/FN per entity type. Computed once; reused 20,000 times.
    ti = {t: i for i, t in enumerate(types)}
    n, T = len(y_true), len(types)
    tp, fp, fn = (np.zeros((n, T)) for _ in range(3))
    for d, (a, b) in enumerate(zip(y_true, y_pred)):
        g, p = span_set(a), span_set(b)
        for e in g & p: tp[d, ti[e[0]]] += 1
        for e in p - g: fp[d, ti[e[0]]] += 1
        for e in g - p: fn[d, ti[e[0]]] += 1
    return tp, fp, fn

def macro_from_counts(tp, fp, fn, idx=None):
    if idx is not None: tp, fp, fn = tp[idx], fp[idx], fn[idx]
    TP, FP, FN = tp.sum(0), fp.sum(0), fn.sum(0)
    z = np.zeros_like(TP)
    prec = np.divide(TP, TP+FP, out=z.copy(), where=(TP+FP) > 0)
    rec  = np.divide(TP, TP+FN, out=z.copy(), where=(TP+FN) > 0)
    f1   = np.divide(2*prec*rec, prec+rec, out=z.copy(), where=(prec+rec) > 0)
    return float(f1.mean())

TYPES = sorted({e[0] for s in yt_t for e in span_set(s)}
               | {e[0] for s in yp_base for e in span_set(s)}
               | {e[0] for s in yp_kd for e in span_set(s)})
CNT_A = count_matrix(yt_t, yp_base, TYPES)
CNT_B = count_matrix(yt_t, yp_kd,   TYPES)

# Sanity: the fast path must agree with seqeval before we trust 20,000 iterations.
_fast, _ref = macro_from_counts(*CNT_B), entity_metrics(yt_t, yp_kd)["macro_f1"]
assert abs(_fast - _ref) < 1e-9, f"fast macro {_fast} != seqeval {_ref}"
print(f"fast macro-F1 verified against seqeval ({_fast:.6f})")

def paired_bootstrap(cnt_a, cnt_b, n=None, seed=42):
    n = n or CFG.bootstrap_n
    rng = np.random.default_rng(seed); N = cnt_a[0].shape[0]
    fa, fb, fd = [], [], []
    for _ in range(n):
        idx = rng.integers(0, N, N)
        a = macro_from_counts(*cnt_a, idx); b = macro_from_counts(*cnt_b, idx)
        fa.append(a); fb.append(b); fd.append(b - a)
    q = lambda v: (float(np.percentile(v, 2.5)), float(np.percentile(v, 97.5)))
    return {"f1_a": macro_from_counts(*cnt_a), "ci_a": q(fa),
            "f1_b": macro_from_counts(*cnt_b), "ci_b": q(fb),
            "delta": macro_from_counts(*cnt_b) - macro_from_counts(*cnt_a),
            "ci_delta": q(fd), "n_resamples": n}

def permutation_test(cnt_a, cnt_b, n=None, seed=42):
    n = n or CFG.permutation_n
    rng = np.random.default_rng(seed); N = cnt_a[0].shape[0]
    obs = macro_from_counts(*cnt_b) - macro_from_counts(*cnt_a)
    hits = 0
    for _ in range(n):
        sw = rng.random(N) < 0.5
        A = tuple(np.where(sw[:, None], y, x) for x, y in zip(cnt_a, cnt_b))
        B = tuple(np.where(sw[:, None], x, y) for x, y in zip(cnt_a, cnt_b))
        if abs(macro_from_counts(*B) - macro_from_counts(*A)) >= abs(obs): hits += 1
    return {"observed_delta": obs, "p_value": (hits+1)/(n+1), "n_permutations": n}

t0 = time.time()
BOOT = paired_bootstrap(CNT_A, CNT_B)
PERM = permutation_test(CNT_A, CNT_B)
print(f"({time.time()-t0:.1f} s for {BOOT['n_resamples']:,} bootstrap + "
      f"{PERM['n_permutations']:,} permutations)\n")

print(f"baseline macro F1 = {BOOT['f1_a']:.4f}  95% CI "
      f"[{BOOT['ci_a'][0]:.4f}, {BOOT['ci_a'][1]:.4f}]")
print(f"KD       macro F1 = {BOOT['f1_b']:.4f}  95% CI "
      f"[{BOOT['ci_b'][0]:.4f}, {BOOT['ci_b'][1]:.4f}]")
print(f"delta = {100*BOOT['delta']:+.2f} pp  95% CI "
      f"[{100*BOOT['ci_delta'][0]:+.2f}, {100*BOOT['ci_delta'][1]:+.2f}] pp")
print(f"paired permutation p = {PERM['p_value']:.4f}")
if BOOT["ci_delta"][0] <= 0 <= BOOT["ci_delta"][1]:
    print("\n[!] The delta CI includes zero. Report it as such -- do not claim a "
          "significant improvement.")

types_w = sorted(set(PER_TYPE["baseline"]) & set(PER_TYPE["kd"]))
fa_ = [PER_TYPE["baseline"][t]["f1-score"] for t in types_w]
fb_ = [PER_TYPE["kd"][t]["f1-score"] for t in types_w]
WILCOXON = {"n_types": len(types_w)}
try:
    if np.allclose(fa_, fb_):
        raise ValueError("all per-type differences are zero")
    w, pv = wilcoxon(fa_, fb_)
    WILCOXON.update({"W": float(w), "p": float(pv)})
    print(f"Wilcoxon over {len(types_w)} types: W={w:.1f}, p={pv:.4f}")
except ValueError as e:
    WILCOXON["error"] = str(e); print("Wilcoxon not computable:", e)

---
## 12. Span-length analysis



In [ ]:
def f1_by_span_length(y_true, y_pred, max_len=4):
    """Strict span-level F1 stratified by gold span length."""
    spans = span_set
    buckets = {}
    for seq_t, seq_p in zip(y_true, y_pred):
        gold, pred = set(spans(seq_t)), set(spans(seq_p))
        for g in gold:
            L = min(g[2]-g[1], max_len)
            b = buckets.setdefault(L, {"tp":0,"fn":0,"fp":0,"n":0})
            b["n"] += 1
            if g in pred: b["tp"] += 1
            else: b["fn"] += 1
        for p in pred:
            L = min(p[2]-p[1], max_len)
            if p not in gold: buckets.setdefault(L, {"tp":0,"fn":0,"fp":0,"n":0})["fp"] += 1
    rows = []
    for L in sorted(buckets):
        b = buckets[L]
        prec = b["tp"]/max(b["tp"]+b["fp"],1); rec = b["tp"]/max(b["tp"]+b["fn"],1)
        rows.append({"span_len": f"{L}+" if L==max_len else str(L), "n_gold": b["n"],
                     "precision": round(prec,4), "recall": round(rec,4),
                     "f1": round(2*prec*rec/max(prec+rec,1e-9), 4)})
    return pd.DataFrame(rows)

LEN_BASE = f1_by_span_length(yt_t, yp_base); LEN_KD = f1_by_span_length(yt_t, yp_kd)
LEN = LEN_BASE.merge(LEN_KD, on=["span_len","n_gold"], suffixes=("_base","_kd"))
LEN["delta_pp"] = (100*(LEN.f1_kd - LEN.f1_base)).round(2)
print(LEN[["span_len","n_gold","f1_base","f1_kd","delta_pp"]].to_string(index=False))
thin = LEN[LEN.n_gold < 100]
if len(thin):
    print("\n[!] Buckets with n<100 -- deltas here are noise, not findings: "
          + ", ".join(thin.span_len))
LEN.to_csv(f"{CFG.out_dir}/span_length.csv", index=False)

---
## 13. Single results file



In [ ]:
def _need(names: Dict[str, str]):
    """Fail before serialising, naming the cell to re-run.

    A NameError at this point would discard hours of work, so check first and
    say exactly what is missing. Long sessions do die mid-run; the teacher cell
    reloads from cache in seconds, so recovery is cheap once the cell to re-run
    is identified.
    """
    missing = [f"  {v}  ->  {k}" for k, v in names.items() if k not in globals()]
    if missing:
        raise RuntimeError("Missing variables. Re-run these cells:\n" + "\n".join(missing))

_need({"TEACHER_STATS": "section 5  (teacher / soft labels)",
       "TEACHER_META":  "section 5  (teacher / soft labels)",
       "GRID":          "section 8  (paired grid)",
       "FINAL":         "section 9  (multi-seed training)",
       "BENCH":         "section 10 (quantisation + latency)",
       "BOOT":          "section 11 (statistics)",
       "PER_TYPE":      "section 9  (multi-seed training)",
       "LEN":           "section 12 (span length)"})

# Teacher stats survive on disk even when the kernel does not.
if "TEACHER_STATS" not in globals() and Path(STATS_PATH).exists():
    TEACHER_STATS = json.loads(Path(STATS_PATH).read_text())
    TEACHER_META = TEACHER_STATS.get("meta", {})

RESULTS = {
    "generated_at": time.strftime("%Y-%m-%dT%H:%M:%S"),
    "environment": ENV,
    "config": {k: (list(v) if isinstance(v, tuple) else v) for k, v in asdict(CFG).items()},
    "label_space": {"granularity": CFG.granularity, "n_labels": N_LABELS, "labels": LABELS},
    "corpus": {"table1": T1.to_dict("records"),
               "entities_train": ENT_TRAIN.to_dict("records"),
               "entities_test": ENT_TEST.to_dict("records")},
    "teacher": {"model": CFG.teacher, **TEACHER_META,
                "fallback_rate": TEACHER_STATS.get("fallback_rate"),
                "positions": TEACHER_STATS.get("positions"),
                "truncated_chunks": TEACHER_STATS.get("truncated_chunks"),
                "decoding": "two-stage constrained (BIO prefix, then type code)"},
    "grid": GRID.to_dict("records"),
    "selected": {"baseline": best_base, "kd": best_kd},
    "final_runs": FINAL,
    "summary": {c: {"macro_f1_mean": float(np.mean([r["macro_f1"] for r in FINAL[c]])),
                    "macro_f1_std": float(np.std([r["macro_f1"] for r in FINAL[c]], ddof=1))
                                    if len(FINAL[c]) > 1 else 0.0,
                    "micro_f1_mean": float(np.mean([r["micro_f1"] for r in FINAL[c]])),
                    "recall_mean": float(np.mean([r["micro_recall"] for r in FINAL[c]])),
                    "n_seeds": len(FINAL[c])} for c in FINAL},
    "rule_baseline": {k: v for k, v in RULE_METRICS.items() if k != "per_type"},
    "rule_baseline_per_type": RULE_METRICS["per_type"],
    "per_type": PER_TYPE,
    "int8": {k: v for k, v in INT8_METRICS.items() if k != "per_type"},
    "benchmark": BENCH,
    "statistics": {"paired_bootstrap": BOOT, "permutation": PERM, "wilcoxon": WILCOXON},
    "span_length": LEN.to_dict("records"),
}
Path(f"{CFG.out_dir}/results.json").write_text(json.dumps(RESULTS, indent=2, default=float))
print(f"written: {CFG.out_dir}/results.json")

print("\n" + "="*66)
print("HEADLINE RESULTS -- single source for every reported number")
print("="*66)
s = RESULTS["summary"]
print(f"Rules only        macro F1 = {RULE_METRICS['macro_f1']:.4f}")
print(f"Baseline (n={s['baseline']['n_seeds']})    macro F1 = {s['baseline']['macro_f1_mean']:.4f} "
      f"+/- {s['baseline']['macro_f1_std']:.4f}")
print(f"KD       (n={s['kd']['n_seeds']})    macro F1 = {s['kd']['macro_f1_mean']:.4f} "
      f"+/- {s['kd']['macro_f1_std']:.4f}")
print(f"KD + INT8         macro F1 = {INT8_METRICS['macro_f1']:.4f}")
print(f"delta KD-baseline = {100*BOOT['delta']:+.2f} pp "
      f"[{100*BOOT['ci_delta'][0]:+.2f}, {100*BOOT['ci_delta'][1]:+.2f}] pp, "
      f"p = {PERM['p_value']:.4f}")
print(f"teacher fallback  = {100*(TEACHER_STATS.get('fallback_rate') or 0):.2f}%")
print(f"INT8 latency      = {BENCH['kd_int8']['latency_ms_mean']:.1f} ms/batch "
      f"@ shape {BENCH['kd_int8']['batch_shape']} on {BENCH['kd_int8']['cpu']}")
print(f"INT8 size         = {BENCH['kd_int8']['size_MB']:.0f} MB")

---
## 14. Figures — generated exclusively from `results.json`

Figures re-read the file from disk rather than using session variables, so a
figure cannot disagree with a table.

In [ ]:
import matplotlib.pyplot as plt
R = json.loads(Path(f"{CFG.out_dir}/results.json").read_text())

fig, ax = plt.subplots(1, 3, figsize=(16, 4.4))

# (a) main comparison with error bars
names = ["Rules", "Baseline", "KD", "KD+INT8"]
vals  = [R["rule_baseline"]["macro_f1"], R["summary"]["baseline"]["macro_f1_mean"],
         R["summary"]["kd"]["macro_f1_mean"], R["int8"]["macro_f1"]]
errs  = [0, R["summary"]["baseline"]["macro_f1_std"], R["summary"]["kd"]["macro_f1_std"], 0]
ax[0].bar(names, vals, yerr=errs, capsize=4,
          color=["#9e9e9e", "#42A5F5", "#26C6DA", "#66BB6A"])
ax[0].set_ylabel("Entity-level macro F1")
ax[0].set_title(f"(a) Test performance (n={R['summary']['kd']['n_seeds']} seeds)")
ax[0].set_ylim(0, 1.0); ax[0].grid(axis="y", alpha=.3)
for i, v in enumerate(vals): ax[0].text(i, v + .02, f"{v:.3f}", ha="center", fontsize=9)

# (b) grid
g = pd.DataFrame(R["grid"]); gk = g[g.model == "kd"]
if len(gk):
    ax[1].scatter(gk["T"], gk.val_macro_f1, c=gk["alpha"], cmap="viridis", s=120,
                  edgecolors="k", linewidth=.5)
    bb = R["selected"]["kd"]
    ax[1].axvline(bb["T"], color="red", ls="--", alpha=.4)
    ax[1].set(xlabel="Temperature T", ylabel="Validation macro F1",
              title=f"(b) KD grid — best T={bb['T']}, α={bb['alpha']}")
    ax[1].grid(alpha=.3)

# (c) span length with support annotated
L = pd.DataFrame(R["span_length"])
x = np.arange(len(L)); w = .38
ax[2].bar(x-w/2, L.f1_base, w, label="Baseline", color="#42A5F5")
ax[2].bar(x+w/2, L.f1_kd,  w, label="KD",       color="#26C6DA")
ax[2].set_xticks(x); ax[2].set_xticklabels(L.span_len)
for i, r in L.iterrows(): ax[2].text(i, .02, f"n={int(r.n_gold)}", ha="center", fontsize=8)
ax[2].set(xlabel="Gold span length (tokens)", ylabel="Span F1",
          title="(c) F1 by span length"); ax[2].legend(); ax[2].grid(axis="y", alpha=.3)

plt.tight_layout(); plt.savefig(f"{CFG.out_dir}/fig_main.pdf", bbox_inches="tight")
plt.savefig(f"{CFG.out_dir}/fig_main.png", dpi=160, bbox_inches="tight"); plt.show()
print("figures written from results.json")

---
## 15. Reproducibility manifest

Freezes library versions, data file hashes, seeds, the full teacher prompt
format and the configuration.

Note the handling of `FEWSHOT`: exemplars are drawn at run time from the
training split, and embedding them here would republish corpus text verbatim.
They are therefore cleared while the prompt template is serialised.

In [ ]:
def teacher_prompt_template() -> str:
    """Serialise the prompt format with a synthetic illustrative sentence.

    FEWSHOT holds real annotated windows from the training split, and
    build_prompt() embeds them verbatim. It is cleared for the duration of this
    call so that the released manifest documents the prompt structure without
    republishing corpus text -- which, on the real partition, would be PHI.
    """
    global FEWSHOT
    saved, FEWSHOT = FEWSHOT, []
    try:
        tok = AutoTokenizer.from_pretrained(CFG.teacher)
        return build_prompt(tok, ["Paciente", "Maria", "Silva", ",", "45", "anos"],
                            build_label_alphabet(tok)[1])
    finally:
        FEWSHOT = saved

def file_hash(p):
    return hashlib.sha256(Path(p).read_bytes()).hexdigest()[:16] if Path(p).exists() else None

MANIFEST = {
    "environment": ENV,
    "config": RESULTS["config"],
    "seeds": list(CFG.seeds),
    "data_files": {n: file_hash(Path(CFG.data_dir)/n) for n in
                   ["clinical_deid_training_set.json","clinical_deid_validation_set.json",
                    "clinical_deid_test_set.json"]},
    "dataset_source": "Venturus/AnonyMED-BR (synthetic partition only; real partition "
                      "withheld pending Ethics Committee approval)",
    "dataset_citation": "Schiezaro M, Rosa G, Pedrini H, Campos BAG. Guardians of the Data: "
                        "NER and LLMs for Effective Medical Record Anonymization in Brazilian "
                        "Portuguese. Front Public Health. 2026;13:1717303.",
    "preprocessing": "Reproduces venturusbr/AnonyMED-BR BERT_fine_tuning.ipynb "
                     "(split_tags + sliding_window + BIO by tag repetition).",
    "teacher_prompt_template": teacher_prompt_template(),
    "teacher_prompt_note": "Few-shot exemplars are drawn at run time from the "
                           "training split by make_fewshot(); they are corpus text "
                           "and are deliberately excluded from this manifest. The "
                           "template above shows the exact prompt structure with a "
                           "synthetic illustrative sentence.",
    "derived_code": "split_tags(), fix_tags(), sliding window and BIO assignment "
                    "are reused from venturusbr/AnonyMED-BR (BERT_fine_tuning.ipynb) "
                    "with attribution; see README.md.",
    "outputs": sorted(p.name for p in Path(CFG.out_dir).glob("*.csv")) +
               ["results.json", "fig_main.pdf"],
}
Path(f"{CFG.out_dir}/manifest.json").write_text(json.dumps(MANIFEST, indent=2, default=str))
print(json.dumps({k: v for k, v in MANIFEST.items()
                  if k != "teacher_prompt_template"}, indent=2, default=str)[:1400])